In [ ]:
import torch
import pandas as pd
from tqdm import tqdm
from Bio import SeqIO

from esm.models.esm3 import ESM3
from esm.sdk.api import ESMProtein, SamplingConfig

def fasta_to_esm3_csv(
    fasta_path: str,
    out_csv: str = "AntiCP-Independent.csv",
    model_name: str = "esm3_sm_open_v1",
    max_len: int = 1022
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    # IMPORTANT: pass registry name, not HF repo
    model = ESM3.from_pretrained(model_name).to(device)
    model.eval()

    rows = []
    records = list(SeqIO.parse(fasta_path, "fasta"))

    for rec in tqdm(records, desc="Embedding proteins"):
        seq_id = rec.id
        seq = str(rec.seq).upper().replace("*", "")

        if len(seq) == 0:
            continue

        if len(seq) > max_len:
            seq = seq[:max_len]

        protein = ESMProtein(sequence=seq)
        protein_tensor = model.encode(protein)

        with torch.no_grad():
            out = model.forward_and_sample(
                protein_tensor,
                SamplingConfig(return_per_residue_embeddings=True)
            )

        # per_residue_embedding includes special tokens; drop first/last then mean-pool
        per_res = out.per_residue_embedding[1:-1] if out.per_residue_embedding.shape[0] >= 3 else out.per_residue_embedding
        emb = per_res.mean(dim=0).detach().to("cpu").float().numpy()

        row = {"id": seq_id, "length": len(seq)}
        for j, v in enumerate(emb):
            row[f"esm3_{j}"] = float(v)
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print("Saved:", out_csv, "| shape:", df.shape)
    return df

# مثال:
df = fasta_to_esm3_csv("AntiCP-Independent.fasta", "esm3_embeddings.csv", model_name="esm3_sm_open_v1")
df.head()
